In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv

from llama_index.readers.file import PDFReader
from llama_index.llms.cohere import Cohere
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document
import pandas as pd

In [ ]:
CSV_PATH = Path('../Data/ChatbotData.csv')

# Cohere API Key는 Git에 올리지 않기 위해 .env 파일에서 읽습니다.
# 프로젝트 루트에 `.env` 파일을 만들고 아래처럼 저장하세요.
# COHERE_API_KEY=your_cohere_api_key
env_path = find_dotenv(filename='.env', usecwd=True)
load_dotenv(env_path)

cohere_api_key = os.getenv('COHERE_API_KEY')
if not cohere_api_key:
    raise ValueError('COHERE_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요.')

Settings.llm = Cohere(
    # model='command-r-08-2024',
    model='command-r7b-12-2024',
    api_key=cohere_api_key,
    temperature=0,  # 낮을수록 일관된 답변을 생성합니다.
)
Settings.embed_model = CohereEmbedding(
    api_key=cohere_api_key,
    model_name='embed-multilingual-v3.0',
    input_type='search_document',
    embed_batch_size=96,
)

print(f'CSV경로: {CSV_PATH.resolve()}')
print('Cohere / LlamaIndex 설정 완료')

#### CSV를 문서 형태로 변환

In [22]:
df = pd.read_csv(CSV_PATH)
df.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [23]:
TEXT_COLUMNS = ['Q','A']
METADATA_COLUMNS = ['label']

MAX_ROWS = 1000
df = df.head(MAX_ROWS).copy()

display(df.head())
print('문서와 대상 컬럼 : ',TEXT_COLUMNS)
print('메타데이터 컬럼 : ',METADATA_COLUMNS)
print('사용할 행수 : ',MAX_ROWS)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


문서와 대상 컬럼 :  ['Q', 'A']
메타데이터 컬럼 :  ['label']
사용할 행수 :  1000


In [24]:
# 각 Row를 질문-답변 형태의 문서로 변환

def row_to_document(row:pd.Series, row_number:int) -> Document:
    text_parts = []

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.isna(value):
            continue
        text_parts.append(f'{column}:{value}')

    metadata = {
        'row_number' : row_number,
        'label' : row['label']
    }
    return Document(
        text = ' | '.join(text_parts),
        metadata = metadata
    )    
# DataFrame의 각 row를 Document로 변환
documents = [row_to_document(row, idx) for idx, row in df.iterrows()]

print('생성된 Document수 : ',len(documents))
print('첫번째 Document 예제')
print(documents[0].text)

생성된 Document수 :  1000
첫번째 Document 예제
Q:12시 땡! | A:하루가 또 가네요.


In [25]:
# 문서목록으로 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

# 검색된 문서를 바탕으로 답변하는 chat engine을 제작
# as_query_engine는 검색하여 답변
# as_chat_engine은 추론
chat_engine = index.as_chat_engine(
    chat_mode='context',
    similartity_top_k = 5,
    verbose = True
)

2026-04-28 11:42:04,997 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:05,548 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:06,022 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:06,803 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:07,431 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:07,931 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:08,483 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:08,987 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:09,514 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:42:09,981 - INFO - HTTP Request: POST https://api.cohere.com/v2/embe

In [28]:
# Test
question = '12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?'
response = chat_engine.chat(question)
print('질문:',question)
print('응답:',response)

2026-04-28 11:44:10,663 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:44:11,999 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


질문: 12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?
응답: "12시 땡! 하루가 또 가네요."라는 질문에는 "가끔 뭐하는지 궁금해"라는 답변이 연결되어 있습니다.


In [30]:
# 여러번 질문하고
# exit, quit 종료
while True:
    user_question = input('질문을 입력하세요:').strip()
    if user_question.lower() in {'exit',quit}:
        print('쳇봇을 종료합니다.')
        break
    if not user_question:
        print('빈 질문은 처리 할 수 없다. 다시 입력하여라')
        continue

    answer = chat_engine.chat(user_question)
    print('\n[응답]')
    print(answer)
    print('-'*50)    


2026-04-28 11:49:31,463 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:49:33,214 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"



[응답]
12시 땡! 하루가 또 가네요. 가끔 뭐하는지 궁금해.
--------------------------------------------------


2026-04-28 11:50:10,180 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:51:30,484 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"



[응답]
시험을 앞두고 계시군요! 시험 준비를 잘하고 계신가요? 시험을 잘 치를 수 있도록 응원할게요! 힘내세요!
--------------------------------------------------


2026-04-28 11:51:51,497 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
2026-04-28 11:51:56,001 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"



[응답]
quit이라는 단어는 "그만두다"라는 의미로 사용될 수 있습니다. 만약 시험 준비를 그만두는 상황이라면, 시험 준비를 계속 이어가고 싶으시다면 다른 방법을 찾아보시는 것이 좋을 것 같습니다. 시험 준비를 계속 이어가고 싶으시다면, 다음과 같은 방법들을 고려해보세요:

1. **시험 범위를 다시 한번 확인**: 시험 범위를 다시 한번 확인하고, 어떤 부분이 부족한지 파악하세요.
2. **시험 범위 내에서 문제 풀기**: 시험 범위 내에서 문제를 풀어보면서 실력을 향상시키세요.
3. **시험 범위 내에서 요약하기**: 시험 범위 내에서 요약을 만들어보면서 이해도를 높여보세요.
4. **시험 범위 내에서 동영상 강의 듣기**: 시험 범위 내에서 동영상 강의를 들어보면서 이해도를 높여보세요.
5. **시험 범위 내에서 모의고사 풀기**: 시험 범위 내에서 모의고사를 풀어보면서 실력을 향상시키세요.
6. **시험 범위 내에서 스터디 그룹 만들기**: 시험 범위 내에서 스터디 그룹을 만들어서 서로 공부하고 서로 격려하세요.

시험 준비를 계속 이어가고 싶으시다면, 위 방법들을 고려해보세요. 힘내세요!
--------------------------------------------------
쳇봇을 종료합니다.
